# taskboard

> The Phase 10 task board: seeded "find x" puzzles with a checker, and a board of expiring subtasks in the Phase 5 world protocol (time in turns, Phase 6 ledger, individual / team / difference rewards, a social-optimum bound); `TaskBoardSim` is the same board for numpy agents, with a fixed solve rate.

In [ ]:
#| default_exp taskboard

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass

import numpy as np

from explRL.multi import spawnRngs

OPS = ('+', '*', '^', '%')


@dataclass
class Puzzle:
    ops: tuple        # ((op, k), ...) applied left to right to x
    hi: int           # answers are integers in [0, hi)
    y: int
    answer: int       # the hidden x* with f(x*) == y (one of possibly several)

    def f(self, x):
        for op, k in self.ops:
            x = x + k if op == '+' else x * k if op == '*' else x ^ k if op == '^' else x % k
        return x

    def check(self, x):
        "Our evaluation of the generated function; model text is never executed."
        return isinstance(x, (int, np.integer)) and not isinstance(x, bool) and 0 <= x < self.hi and self.f(int(x)) == self.y

    def text(self):
        e = 'x'
        for op, k in self.ops:
            e = f'({e} {op} {k})'
        return f'def f(x): return {e[1:-1]}\nFind an integer x with 0 <= x < {self.hi} and f(x) == {self.y}.'


def makePuzzle(rng, difficulty):
    "`difficulty + 1` operations with small constants, answer range [0, 50 * 2**difficulty)."
    ops = []
    for _ in range(int(difficulty) + 1):
        op = OPS[int(rng.integers(4))]
        k = int(rng.integers(1, 20)) if op == '+' else int(rng.integers(2, 10)) if op == '*' else \
            int(rng.integers(1, 64)) if op == '^' else int(rng.integers(50, 200))
        ops.append((op, k))
    hi = 50 * 2 ** int(difficulty)
    p = Puzzle(tuple(ops), hi, 0, int(rng.integers(hi)))
    p.y = p.f(p.answer)
    return p


@dataclass
class Task:
    id: int
    value: float
    puzzle: Puzzle
    appear: float
    expire: float
    solvedBy: int = -1


class TaskBoardEnv:
    """The task board in the Phase 5 world protocol, time in turns: every action takes 1. `2 nAgents + 2` subtasks are
    open; each expires `life` turns after it appears and is replaced at once, as is a solved one. Actions:
    ('open', task id), ('submit', answer), ('wait', None). Ledger rows use the Phase 6 format."""
    timeUnit = 'turn'

    def __init__(self, seed, nAgents, difficulty=2, value=(1, 2, 3), life=30, penalty=0.5, scheme='individual'):
        self.difficulty, self.values, self.life = int(difficulty), tuple(value), float(life)
        self.penalty, self.scheme = float(penalty), scheme
        self.reset(seed, nAgents)

    def reset(self, seed, nAgents):
        self.rng, self.rngSim = spawnRngs(seed, 2)
        self.nAgents, self.tWorld, self.serial = int(nAgents), 0.0, 0
        self.tasks, self.open, self.ledger, self.netRows = {}, [], [], []
        self.opened, self.last = [None] * nAgents, [0.0] * nAgents
        self.pending, self.wrongBy = np.zeros(nAgents), np.zeros(nAgents, int)
        self.counts = dict(solved=0, wrong=0, duplicates=0, opens=0)
        M = 2 * nAgents + 2
        for k in range(M):          # staggered ages, so expiries spread over the first `life` turns
            self._new(-self.life * k / M)

    def _new(self, t):
        v = float(self.values[int(self.rng.integers(len(self.values)))])
        task = Task(self.serial, v, makePuzzle(self.rng, self.difficulty), t, t + self.life)
        self.tasks[task.id] = task
        self.open.append(task.id)
        self.serial += 1

    def advance(self, t):
        if t < self.tWorld:
            raise ValueError('time went backwards')
        while True:
            due = [k for k in self.open if self.tasks[k].expire <= t]
            if not due:
                break
            k = min(due, key=lambda j: (self.tasks[j].expire, j))
            self.open.remove(k)
            self._new(self.tasks[k].expire)
        self.tWorld = float(t)

    def _close(self, k, t):
        self.open.remove(k)
        self._new(t)

    def _reward(self, i, delta):
        if self.scheme != 'team':        # 'individual', and 'difference' (equal here: only the first answer scores)
            return delta
        self.pending += delta / self.nAgents
        r, self.pending[i] = float(self.pending[i]), 0.0
        return r

    def apply(self, i, action, t):
        kind_, arg = action
        self.advance(t)
        k, h, delta, ok = self.opened[i], 0.0, 0.0, None
        if kind_ == 'open' and arg in self.open:
            self.opened[i], self.counts['opens'] = int(arg), self.counts['opens'] + 1
            row = (i, t, 1.0, 'travel', int(arg), None, 0.0, 0.0, float('nan'), float('nan'))
            info = dict(kind='travel', harvest=0.0, nHere=0, patch=None, dur=1.0, task=int(arg), result='opened')
        elif kind_ == 'submit' and k is not None:
            task, ok = self.tasks[k], self.tasks[k].puzzle.check(arg)
            if not ok:
                delta, res = -self.penalty, 'wrong'
                self.counts['wrong'] += 1
                self.wrongBy[i] += 1
            elif k in self.open:
                h = delta = task.value
                task.solvedBy, res = i, 'solved'
                self.counts['solved'] += 1
                self._close(k, t)
            else:
                res = 'late'
                self.counts['duplicates'] += 1
            row = (i, t, 1.0, 'stay', k, k if h > 0 else None, h, 0.0, t, t + 1.0)
            info = dict(kind='stay', harvest=h, nHere=1, patch=k, dur=1.0, task=k, result=res)
        else:
            row = (i, t, 1.0, 'stay', -1, None, 0.0, 0.0, t, t + 1.0)
            info = dict(kind='stay', harvest=0.0, nHere=0, patch=None, dur=1.0, task=None,
                        result='wait' if kind_ == 'wait' else 'invalid')
        info['net'] = delta
        r = self._reward(i, delta)
        self.ledger.append(row[:7] + (r,) + row[8:])
        self.netRows.append(delta)          # net value of this row, whatever the reward scheme (team totals)
        self.last[i] = r
        return r, 1.0, row[4], info

    def observe(self, i):
        return dict(time=self.tWorld, opened=self.opened[i], lastReward=self.last[i],
                    open=[(k, self.tasks[k].value, self.tasks[k].expire) for k in self.open])

    def neighbours(self, i, radius):
        return [k for k in range(self.nAgents) if k != i]

    def socialOptimum(self, solveRate=1.0):
        "Upper bound on team value per turn: agents on the highest-value distinct open tasks, two turns per attempt."
        v = sorted((self.tasks[k].value for k in self.open), reverse=True)[:self.nAgents]
        return solveRate * sum(v) / 2

    def netByAgent(self):
        "Value solved minus penalties paid, per agent (the welfare measure; independent of the reward scheme)."
        h = np.bincount([t.solvedBy for t in self.tasks.values() if t.solvedBy >= 0],
                        weights=[t.value for t in self.tasks.values() if t.solvedBy >= 0], minlength=self.nAgents)
        return h - self.penalty * self.wrongBy

    def duplicateTurns(self):
        "Turns spent (opening or submitting) on tasks that another agent solved: duplicate work."
        return sum(1 for r in self.ledger if r[4] >= 0 and self.tasks[r[4]].solvedBy not in (-1, r[0]))

    def firstSolves(self):
        "Solves (first correct answers) per agent: the board's discoveries."
        return np.bincount([t.solvedBy for t in self.tasks.values() if t.solvedBy >= 0], minlength=self.nAgents)


class TaskBoardSim(TaskBoardEnv):
    "The same board for numpy agents: ('submit', None) succeeds with probability `solveRate`."
    def __init__(self, seed, nAgents, solveRate=0.5, **kw):
        self.solveRate = float(solveRate)
        super().__init__(seed, nAgents, **kw)

    def apply(self, i, action, t):
        if action[0] == 'submit' and action[1] is None and self.opened[i] is not None:
            p = self.tasks[self.opened[i]].puzzle
            action = ('submit', p.answer if self.rngSim.random() < self.solveRate else -1)
        return super().apply(i, action, t)

## Tests

In [ ]:
import numpy as np
from explRL.taskboard import *

# puzzles: the same seed gives the same puzzle; the stated answer is in range and satisfies f at every difficulty
for d in range(6):
    p, q = makePuzzle(np.random.default_rng(d), d), makePuzzle(np.random.default_rng(d), d)
    assert p == q and len(p.ops) == d + 1 and p.hi == 50 * 2 ** d and p.check(p.answer)
    # the checker accepts exactly the in-range x with f(x) == y
    ok = {x for x in range(p.hi) if p.f(x) == p.y}
    assert {x for x in range(-5, p.hi + 5) if p.check(x)} == ok and p.answer in ok
    assert not p.check(True) and not p.check('3') and not p.check(float(p.answer))
p = Puzzle((('*', 7), ('+', 3), ('^', 21), ('%', 97)), 400, 0, 5); p.y = p.f(5)
assert p.text().startswith('def f(x): return (((x * 7) + 3) ^ 21) % 97') and f'== {p.y}' in p.text()
assert eval(p.text().split('return ')[1].split('\n')[0], {'x': 5}) == p.y      # the text is the function we check

# the board: 2N + 2 open subtasks; staggered expiries; replaced on expiry
env = TaskBoardEnv(0, 2, difficulty=1, life=10)
assert len(env.open) == 6 and env.observe(0)['opened'] is None
first = sorted(env.tasks[k].expire for k in env.open)
assert first == sorted(10 - 10 * k / 6 for k in range(6))
env.advance(5.0)
assert len(env.open) == 6 and env.serial == 9            # the three that expired by t = 5 (at 1.67, 3.33, 5) replaced
assert all(env.tasks[k].expire > 5.0 for k in env.open)

# opening costs a turn; only the first correct answer scores; wrong answers pay the penalty
env = TaskBoardEnv(1, 2, difficulty=1, life=100, penalty=0.5)
k = env.open[0]; task = env.tasks[k]
r, dt, loc, info = env.apply(0, ('open', k), 0.0)
assert (r, dt, loc, info['kind'], info['result']) == (0.0, 1.0, k, 'travel', 'opened') and env.observe(0)['opened'] == k
env.apply(1, ('open', k), 0.0)
r, *_ , info = env.apply(0, ('submit', task.puzzle.answer + task.puzzle.hi), 1.0)
assert r == -0.5 and info['result'] == 'wrong'
r, _, _, info = env.apply(0, ('submit', task.puzzle.answer), 2.0)
assert r == task.value and info['result'] == 'solved' and k not in env.open and len(env.open) == 6
r, _, _, info = env.apply(1, ('submit', task.puzzle.answer), 2.0)
assert r == 0.0 and info['result'] == 'late' and env.counts == dict(solved=1, wrong=1, duplicates=1, opens=2)
assert env.apply(1, ('wait', None), 3.0)[0] == 0.0 and env.apply(1, ('open', 999), 4.0)[3]['result'] == 'invalid'
assert env.duplicateTurns() == 2 and len(env.netRows) == len(env.ledger) and np.isclose(sum(env.netRows), env.netByAgent().sum())                          # agent 1 opened and answered agent 0's task
assert list(env.firstSolves()) == [1, 0] and np.allclose(env.netByAgent(), [task.value - 0.5, 0])
L = env.ledger
assert [x[3] for x in L] == ['travel', 'travel', 'stay', 'stay', 'stay', 'stay', 'stay'] and L[3][6] == task.value

# reward schemes: team shares every solve and penalty equally; totals match individual
def play(scheme):
    e = TaskBoardEnv(2, 2, difficulty=0, life=100, scheme=scheme)
    rs = []
    for t, (i, a) in enumerate([(0, ('open', e.open[0])), (1, ('open', e.open[1])), (0, 'ans'), (1, ('submit', -1)),
                                (0, ('wait', None)), (1, ('wait', None))]):
        if a == 'ans':
            a = ('submit', e.tasks[e.opened[0]].puzzle.answer)
        rs.append(e.apply(i, a, float(t))[0])
    return rs, e
ind, e1 = play('individual'); team, e2 = play('team'); diff, _ = play('difference')
assert diff == ind and np.isclose(sum(ind), sum(team)) and np.allclose(e1.netByAgent(), e2.netByAgent())
assert team[2] == e1.tasks[0].value / 2 if e1.tasks[0].solvedBy == 0 else True

# social optimum: top-N open values, two turns per attempt
env = TaskBoardEnv(3, 2, difficulty=1)
top = sorted((env.tasks[k].value for k in env.open), reverse=True)[:2]
assert env.socialOptimum(0.5) == 0.5 * sum(top) / 2

# the numpy board: submissions succeed with the given rate
sim = TaskBoardSim(4, 1, solveRate=1.0, difficulty=3)
sim.apply(0, ('open', sim.open[0]), 0.0)
assert sim.apply(0, ('submit', None), 1.0)[3]['result'] == 'solved'
sim = TaskBoardSim(4, 1, solveRate=0.0, difficulty=3)
sim.apply(0, ('open', sim.open[0]), 0.0)
assert sim.apply(0, ('submit', None), 1.0)[3]['result'] == 'wrong'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()